# Sail Lab: getting started

This notebook talks to the Sail server over Spark Connect (`SPARK_REMOTE=sc://sail:50051`). There is no JVM here: PySpark is only a client, Sail does the work.

Sessions from this notebook show up on the Grafana **Overview** dashboard (operators, spill, CPU). For a per-run view, submit the same code as a job with `./lab submit`.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.getOrCreate()
spark.range(5).show()

In [ ]:
# Sail's plans are DataFusion operators, not Spark's
df = spark.range(1_000_000).select((F.col('id') % 10).alias('k'), F.rand(seed=1).alias('v'))
df.groupBy('k').agg(F.avg('v')).explain()

In [ ]:
# Data written by the jobs lives in MinIO (run ./lab submit gen_orders.py first)
orders = spark.read.parquet('s3://lake/raw/orders')
orders.groupBy('status').count().orderBy('status').show()

In [ ]:
# Delta Lake tables written by jobs/delta_lake.py, including time travel
spark.read.format('delta').option('versionAsOf', 0).load('s3://lake/delta/accounts').count()